In [1]:
from ultralytics import YOLO

In [ ]:
# model = YOLO("yolov8n.pt")

In [5]:
results = model("img_src/traffic_image.webp")
results[0].show()


image 1/1 /Users/deepakpraveen/Documents/project/NLP/img_src/traffic_image.webp: 384x640 4 cars, 1 bus, 1 truck, 23.7ms
Speed: 1.7ms preprocess, 23.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)


In [6]:
# extract vehicle count

count = 0
for r in results:
    for obj in r.boxes.cls:
        if int(obj) in [2,3,5,7]:  # car, bus, truck, motorcycle indexes
            count += 1

print("Detected vehicles:", count)


Detected vehicles: 6


In [9]:
# Speed Estimation (Optical Flow)

import cv2
import numpy as np

cap = cv2.VideoCapture("img_src/traffic_video.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)

# First frame
ret, prev = cap.read()
prev_gray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)
points = cv2.goodFeaturesToTrack(prev_gray, 200, 0.01, 7)

speeds = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    new_points, status, err = cv2.calcOpticalFlowPyrLK(prev_gray, gray, points, None)

    movement = np.sqrt(np.sum((new_points - points)**2, axis=2))
    pixel_speed = np.mean(movement)

    meter_per_pixel = 0.05  
    kmh = (pixel_speed * meter_per_pixel * fps) * 3.6  
    speeds.append(kmh)

    prev_gray = gray.copy()
    points = new_points

print("Estimated Average Speed:", np.mean(speeds))


Estimated Average Speed: 2.7572389


In [10]:
# Congestion logic

if count > 10 and np.mean(speeds) < 20:
    print("Camera Congestion Detected")
else:
    print("Camera traffic Normal")


Camera traffic Normal
